# Testprocedures SDM – GreatOutdoors

Dit notebook valideert of de ETL-pipeline wijzigingen uit de operationele databases correct doorvoert in het SDM.

| Cel | Inhoud |
|-----|--------|
| 1 | Setup |
| 2 | **Test 1** – Update: wijziging in bron wordt overgenomen in SDM |
| 3 | **Test 2** – Verwijdering: verwijderd record verdwijnt uit SDM |
| 4 | SDM herstellen naar originele toestand |

In [ ]:
# ══════════════════════════════════════════════
# CEL 1 – SETUP
# ══════════════════════════════════════════════

import sqlite3
import pandas as pd
from pathlib import Path
from io import StringIO
import sys

# Paden
SDM_PATH    = "../data/sdm_deproject.db"
SOURCE_PATH = "../GreatOutdoors/GreatOutdoors/GO_SALES-data.sqlite"

# Welk record wordt getest
TEST_TABLE  = "order_method"
TEST_PK     = "order_method_code"
TEST_PK_VAL = 1
TEST_COL    = "order_method_en"

SQLITE_SOURCES = {
    "../GreatOutdoors/GreatOutdoors/GO_SALES-data.sqlite": [
        "order_header", "order_details", "order_method", "product_line",
        "product_type", "product", "sales_branch", "sales_staff",
        "retailer_site", "returned_item", "return_reason", "country",
    ],
    "../GreatOutdoors/GreatOutdoors/CRM-data.sqlite": [
        "customer", "customer_type", "customer_store", "customer_headquarters",
        "customer_segment", "customer_contact", "crm_country", "sales_territory",
        "age_group", "sales_demographic",
    ],
    "../GreatOutdoors/GreatOutdoors/GO_STAFF-data.sqlite": [
        "sales_representative", "sales_office", "course", "training",
        "satisfaction_type", "satisfaction",
    ],
}
CSV_SOURCES = {
    "../GreatOutdoors/GreatOutdoors/SALES_TARGET-data.csv":     "sales_target",
    "../GreatOutdoors/GreatOutdoors/PRODUCT_FORECAST-data.csv": "product_forecast",
    "../GreatOutdoors/GreatOutdoors/INVENTORY_LEVELS-data.csv": "inventory_levels",
}


# ── ETL-functies (stil: geen output) ──
def _load_from_sqlite(sdm_con, source_path, tables):
    if not Path(source_path).exists():
        return
    src_con = sqlite3.connect(source_path)
    for table in tables:
        df = pd.read_sql(f"SELECT * FROM {table}", src_con)
        sdm_cols = [row[1] for row in sdm_con.execute(f"PRAGMA table_info({table})")]
        df.columns = df.columns.str.upper()
        sdm_cols_upper = [c.upper() for c in sdm_cols]
        df = df[[col for col in df.columns if col in sdm_cols_upper]]
        df.columns = [sdm_cols[sdm_cols_upper.index(col)] for col in df.columns]
        df.to_sql(table, sdm_con, if_exists="append", index=False)
    src_con.close()

def _load_from_csv(sdm_con, csv_path, table):
    if not Path(csv_path).exists():
        return
    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    df = df.drop_duplicates()
    df.to_sql(table, sdm_con, if_exists="append", index=False)

def etl_reload():
    """Leeg het SDM en laad alle brondata opnieuw in (stil)."""
    sdm_con = sqlite3.connect(SDM_PATH)
    all_tables = [t for tables in SQLITE_SOURCES.values() for t in tables] + list(CSV_SOURCES.values())
    for table in all_tables:
        sdm_con.execute(f"DELETE FROM {table}")
    for source_path, tables in SQLITE_SOURCES.items():
        _load_from_sqlite(sdm_con, source_path, tables)
    for csv_path, table in CSV_SOURCES.items():
        _load_from_csv(sdm_con, csv_path, table)
    sdm_con.commit()
    sdm_con.close()


# ── Leeshelperfuncties ──
def lees_uit_bron(tabel, pk_kolom, pk_waarde, kolom):
    con = sqlite3.connect(SOURCE_PATH)
    result = con.execute(f"SELECT {kolom} FROM {tabel} WHERE {pk_kolom} = ?", (pk_waarde,)).fetchone()
    con.close()
    return result[0] if result else None

def lees_uit_sdm(tabel, pk_kolom, pk_waarde, kolom):
    con = sqlite3.connect(SDM_PATH)
    result = con.execute(f"SELECT {kolom} FROM {tabel} WHERE {pk_kolom} = ?", (pk_waarde,)).fetchone()
    con.close()
    return result[0] if result else None


print("✓ Setup klaar")
print(f"  Testobject : tabel '{TEST_TABLE}', kolom '{TEST_COL}', PK = {TEST_PK_VAL}")
print(f"  Huidige waarde in bron : {lees_uit_bron(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)}")
print(f"  Huidige waarde in SDM  : {lees_uit_sdm(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)}")

✓ Setup klaar
  Testobject : tabel 'order_method', kolom 'order_method_en', PK = 1
  Huidige waarde in bron : Fax
  Huidige waarde in SDM  : Fax


In [6]:
# ══════════════════════════════════════════════
# CEL 2 – TEST 1: UPDATE
#
# Scenario: een waarde wordt gewijzigd in de
# operationele database. Na het uitvoeren van
# de ETL moet de nieuwe waarde ook in het SDM
# zichtbaar zijn.
# ══════════════════════════════════════════════

print("TEST 1 - UPDATE")

# ── Stap 1: originele waarde vastleggen ──
origineel = lees_uit_bron(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)
print(f"\nSTAP 1 – Originele waarde in bron  : '{origineel}'")
print(f"         Huidige waarde in SDM      : '{lees_uit_sdm(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)}'")

# ── Stap 2: waarde aanpassen in de bron ──
nieuwe_waarde = "TEST_UPDATE_WAARDE"
con = sqlite3.connect(SOURCE_PATH)
con.execute(f"UPDATE {TEST_TABLE} SET {TEST_COL} = ? WHERE {TEST_PK} = ?", (nieuwe_waarde, TEST_PK_VAL))
con.commit()
con.close()
print(f"\nSTAP 2 – Waarde in bron aangepast naar : '{nieuwe_waarde}'")

# ── Stap 3: ETL uitvoeren ──
print("\nSTAP 3 – ETL uitvoeren (laden uit bron naar SDM)... ", end="")
etl_reload()
print("klaar")

# ── Stap 4: resultaat controleren ──
waarde_in_sdm = lees_uit_sdm(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)
print(f"\nSTAP 4 – Waarde in SDM na ETL      : '{waarde_in_sdm}'")

print()
if waarde_in_sdm == nieuwe_waarde:
    print("✅ GESLAAGD – Update correct overgenomen in SDM")
else:
    print(f"❌ MISLUKT  – SDM heeft '{waarde_in_sdm}', verwacht '{nieuwe_waarde}'")

# ── Stap 5: bron herstellen ──
con = sqlite3.connect(SOURCE_PATH)
con.execute(f"UPDATE {TEST_TABLE} SET {TEST_COL} = ? WHERE {TEST_PK} = ?", (origineel, TEST_PK_VAL))
con.commit()
con.close()
print(f"\n↩ Brondata hersteld naar : '{origineel}'")

TEST 1 - UPDATE

STAP 1 – Originele waarde in bron  : 'Fax'
         Huidige waarde in SDM      : 'Fax'

STAP 2 – Waarde in bron aangepast naar : 'TEST_UPDATE_WAARDE'

STAP 3 – ETL uitvoeren (laden uit bron naar SDM)... klaar

STAP 4 – Waarde in SDM na ETL      : 'TEST_UPDATE_WAARDE'

✅ GESLAAGD – Update correct overgenomen in SDM

↩ Brondata hersteld naar : 'Fax'


In [7]:
# ══════════════════════════════════════════════
# CEL 3 – TEST 2: VERWIJDERING
#
# Scenario: een record wordt verwijderd uit de
# operationele database. Na het uitvoeren van
# de ETL mag het record niet meer in het SDM
# voorkomen.
# ══════════════════════════════════════════════

print("TEST 2 - VERWIJDERING")


# ── Stap 1: record vastleggen voor herstel ──
con = sqlite3.connect(SOURCE_PATH)
backup = pd.read_sql(f"SELECT * FROM {TEST_TABLE} WHERE {TEST_PK} = ?", con, params=(TEST_PK_VAL,))
con.close()
waarde_voor = lees_uit_bron(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)
print(f"\nSTAP 1 – Record in bron  : {TEST_PK} = {TEST_PK_VAL}, {TEST_COL} = '{waarde_voor}'")
print(f"         Record in SDM   : {TEST_PK} = {TEST_PK_VAL}, {TEST_COL} = '{lees_uit_sdm(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)}'")

# ── Stap 2: record verwijderen uit bron ──
con = sqlite3.connect(SOURCE_PATH)
con.execute(f"DELETE FROM {TEST_TABLE} WHERE {TEST_PK} = ?", (TEST_PK_VAL,))
con.commit()
con.close()
print(f"\nSTAP 2 – Record verwijderd uit bron ({TEST_PK} = {TEST_PK_VAL})")

# ── Stap 3: ETL uitvoeren ──
print("\nSTAP 3 – ETL uitvoeren (laden uit bron naar SDM)... ", end="")
etl_reload()
print("klaar")

# ── Stap 4: resultaat controleren ──
waarde_in_sdm = lees_uit_sdm(TEST_TABLE, TEST_PK, TEST_PK_VAL, TEST_COL)
print(f"\nSTAP 4 – Record in SDM na ETL      : {waarde_in_sdm}")

print()
if waarde_in_sdm is None:
    print("✅ GESLAAGD – Verwijdering correct overgenomen in SDM")
else:
    print(f"❌ MISLUKT  – Record staat nog in SDM met waarde '{waarde_in_sdm}'")

# ── Stap 5: bron herstellen ──
con = sqlite3.connect(SOURCE_PATH)
backup.to_sql(TEST_TABLE, con, if_exists="append", index=False)
con.commit()
con.close()
print(f"\n↩ Brondata hersteld : record '{waarde_voor}' teruggezet")

TEST 2 - VERWIJDERING

STAP 1 – Record in bron  : order_method_code = 1, order_method_en = 'Fax'
         Record in SDM   : order_method_code = 1, order_method_en = 'TEST_UPDATE_WAARDE'

STAP 2 – Record verwijderd uit bron (order_method_code = 1)

STAP 3 – ETL uitvoeren (laden uit bron naar SDM)... klaar

STAP 4 – Record in SDM na ETL      : None

✅ GESLAAGD – Verwijdering correct overgenomen in SDM

↩ Brondata hersteld : record 'Fax' teruggezet


In [8]:
# ══════════════════════════════════════════════
# CEL 4 – SDM HERSTELLEN
#
# Synchroniseer het SDM opnieuw met de bron,
# zodat het SDM weer volledig up-to-date is
# na de tests.
# ══════════════════════════════════════════════

print("SDM herstellen naar originele toestand... ", end="")
etl_reload()
print("klaar")

# Bevestiging
con = sqlite3.connect(SDM_PATH)
aantal = con.execute(f"SELECT COUNT(*) FROM {TEST_TABLE}").fetchone()[0]
waarde = con.execute(f"SELECT {TEST_COL} FROM {TEST_TABLE} WHERE {TEST_PK} = ?", (TEST_PK_VAL,)).fetchone()[0]
con.close()

print(f"\n✓ SDM gesynchroniseerd")
print(f"  {TEST_TABLE} bevat {aantal} rijen")
print(f"  {TEST_PK} = {TEST_PK_VAL} → {TEST_COL} = '{waarde}'")

SDM herstellen naar originele toestand... klaar

✓ SDM gesynchroniseerd
  order_method bevat 7 rijen
  order_method_code = 1 → order_method_en = 'Fax'
